In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import (
    PermissionsChange,
    Privilege,
    SecurableType,
)

w = WorkspaceClient()

# TODO: set your catalog name
CATALOG = "fk_ce_2026_07_1382"

# ── fk_analysts ─────────────────────────────────────────────
# USE CATALOG on the catalog
w.grants.update(
    securable_type="catalog",
    full_name=CATALOG,
    changes=[PermissionsChange(
        principal="fk_analysts",
        add=[Privilege.USE_CATALOG],
    )],
)

# USE SCHEMA + SELECT on gold only
w.grants.update(
    securable_type="schema",
    full_name=f"{CATALOG}.gold",
    changes=[PermissionsChange(
        principal="fk_analysts",
        add=[Privilege.USE_SCHEMA, Privilege.SELECT],
    )],
)

# ── fk_engineers ────────────────────────────────────────────
# USE CATALOG on the catalog
w.grants.update(
    securable_type="catalog",
    full_name=CATALOG,
    changes=[PermissionsChange(
        principal="fk_engineers",
        add=[Privilege.USE_CATALOG],
    )],
)

# USE SCHEMA + SELECT + MODIFY on bronze, silver, gold, ops
for schema in ["bronze", "silver", "gold", "ops"]:
    w.grants.update(
        securable_type="schema",
        full_name=f"{CATALOG}.{schema}",
        changes=[PermissionsChange(
            principal="fk_engineers",
            add=[Privilege.USE_SCHEMA, Privilege.SELECT, Privilege.MODIFY],
        )],
    )

# READ VOLUME + WRITE VOLUME on volume landing.raw
w.grants.update(
    securable_type="volume",
    full_name=f"{CATALOG}.landing.raw",
    changes=[PermissionsChange(
        principal="fk_engineers",
        add=[Privilege.READ_VOLUME, Privilege.WRITE_VOLUME],
    )],
)

print("Grants applied successfully.")

In [0]:
# G2–G5: Governance – Entitlements, Column Masks, Row Filter, Evidence
# ================================================================

CATALOG = "fk_ce_2026_07_1382"

# ── G2: Entitlement table ────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.user_entitlements (
  user_email    STRING,
  allowed_city  STRING,
  can_view_pii  BOOLEAN
)
""")

# ── G3: Column-mask functions for gold.dim_customers ──────────
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.ops.mask_email(email STRING)
RETURNS STRING
RETURN CASE
  WHEN is_member('fk_admins') THEN email
  WHEN EXISTS (
    SELECT 1 FROM {CATALOG}.ops.user_entitlements ue
    WHERE ue.user_email = current_user()
      AND ue.can_view_pii = TRUE
  ) THEN email
  ELSE CONCAT(
    SUBSTRING(SPLIT_PART(email, '@', 1), 1, 1),
    '***@',
    SPLIT_PART(email, '@', 2)
  )
END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.ops.mask_phone(phone STRING)
RETURNS STRING
RETURN CASE
  WHEN is_member('fk_admins') THEN phone
  WHEN EXISTS (
    SELECT 1 FROM {CATALOG}.ops.user_entitlements ue
    WHERE ue.user_email = current_user()
      AND ue.can_view_pii = TRUE
  ) THEN phone
  ELSE CONCAT('XXXXXX', RIGHT(CAST(phone AS STRING), 4))
END
""")

spark.sql(f"ALTER TABLE {CATALOG}.gold.dim_customers ALTER COLUMN email SET MASK {CATALOG}.ops.mask_email")
spark.sql(f"ALTER TABLE {CATALOG}.gold.dim_customers ALTER COLUMN phone SET MASK {CATALOG}.ops.mask_phone")

# ── G4: Row-filter function for gold.agg_daily_store_sales ──
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.ops.filter_city(city STRING)
RETURNS BOOLEAN
RETURN CASE
  WHEN is_member('fk_admins') THEN TRUE
  WHEN NOT EXISTS (
    SELECT 1 FROM {CATALOG}.ops.user_entitlements ue
    WHERE ue.user_email = current_user()
  ) THEN FALSE
  WHEN EXISTS (
    SELECT 1 FROM {CATALOG}.ops.user_entitlements ue
    WHERE ue.user_email = current_user()
      AND ue.allowed_city = 'ALL'
  ) THEN TRUE
  ELSE city IN (
    SELECT ue.allowed_city FROM {CATALOG}.ops.user_entitlements ue
    WHERE ue.user_email = current_user()
  )
END
""")

spark.sql(f"ALTER TABLE {CATALOG}.gold.agg_daily_store_sales SET ROW FILTER {CATALOG}.ops.filter_city ON (city)")

# ── G5: Evidence and persistence ─────────────────────────────
my_email = spark.sql("SELECT current_user() AS u").collect()[0][0]

# Set own entitlement to Jaipur, can_view_pii = FALSE
spark.sql(f"TRUNCATE TABLE {CATALOG}.ops.user_entitlements")
spark.sql(f"INSERT INTO {CATALOG}.ops.user_entitlements VALUES ('{my_email}', 'Jaipur', FALSE)")

# Create evidence table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.governance_evidence (
  evidence_name   STRING,
  observed_value  STRING
)
""")

# Query protected tables as yourself and capture evidence
row_count = spark.sql(f"SELECT COUNT(*) AS c FROM {CATALOG}.gold.agg_daily_store_sales").collect()[0][0]
masked_email = spark.sql(f"SELECT email AS e FROM {CATALOG}.gold.dim_customers WHERE customer_id = 'C0001'").collect()[0][0]
masked_phone = spark.sql(f"SELECT phone AS p FROM {CATALOG}.gold.dim_customers WHERE customer_id = 'C0001'").collect()[0][0]

spark.sql(f"TRUNCATE TABLE {CATALOG}.ops.governance_evidence")
spark.sql(f"""
INSERT INTO {CATALOG}.ops.governance_evidence VALUES
  ('ROW_FILTER_ROW_COUNT_JAIPUR', '{row_count}'),
  ('MASKED_EMAIL_C0001',          '{masked_email}'),
  ('MASKED_PHONE_C0001',          '{masked_phone}')
""")

# Change entitlement to ALL, can_view_pii = FALSE  (table ends with exactly one row: yours)
spark.sql(f"""
UPDATE {CATALOG}.ops.user_entitlements
SET allowed_city = 'ALL', can_view_pii = FALSE
WHERE user_email = '{my_email}'
""")

print("G2–G5 applied successfully.")
print(f"\nEvidence captured (entitlement was Jaipur / can_view_pii=FALSE):")
display(spark.sql(f"SELECT * FROM {CATALOG}.ops.governance_evidence"))
print(f"\nFinal entitlement row:")
display(spark.sql(f"SELECT * FROM {CATALOG}.ops.user_entitlements"))